In [19]:
# put this near the top of project.ipynb
from pathlib import Path
import os

# Resolve ROOT no matter where the notebook runs from
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()

# Robust dotenv load: point directly at the root .env
from dotenv import load_dotenv
load_dotenv(dotenv_path=ROOT / ".env", override=True)

print("cwd =", os.getcwd())
print("ROOT .env exists? ->", (ROOT / ".env").exists())
print("FMP_API_KEY loaded? ->", bool(os.getenv("FMP_API_KEY")))


cwd = C:\Users\User\bootcamp_Khushi_Khanna\project\notebooks
ROOT .env exists? -> True
FMP_API_KEY loaded? -> True


In [20]:
from pathlib import Path
import pandas as pd, numpy as np, os, datetime as dt, yaml, requests

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
CFG  = ROOT / "project" / "config" / "model_v2.yml"
with open(CFG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

RAW_DIR = ROOT / cfg["paths"]["raw_dir"]; RAW_DIR.mkdir(parents=True, exist_ok=True)

BANK_TICKERS = {
    "JP Morgan":       "JPM",
    "Deutsche Bank":   "DB",
    "Morgan Stanley":  "MS",
    "Bank of America": "BAC",
    "Goldman Sachs":   "GS",
}

def ts(): return dt.datetime.now().strftime("%Y%m%d-%H%M")


In [21]:
# load .env
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

FMP_API_KEY = os.getenv("FMP_API_KEY", "").strip()
assert FMP_API_KEY, "FMP_API_KEY missing. Create .env with FMP_API_KEY=... and restart kernel."

# 5-year window (20 quarters)
end_q   = pd.Period(pd.Timestamp.today(), freq="Q").to_timestamp("Q")
start_q = (end_q - pd.offsets.DateOffset(years=5)).to_period("Q").to_timestamp("Q")
quarters = pd.period_range(start=start_q, end=end_q, freq="Q").to_timestamp("Q")

def to_quarter_end(d): 
    return pd.PeriodIndex(pd.to_datetime(d), freq="Q").to_timestamp("Q")

def within_5y(df):
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    return d[(d["date"] >= start_q) & (d["date"] <= end_q)]


In [22]:
def http_get_json(url, params=None, timeout=30):
    r = requests.get(url, params=params or {}, timeout=timeout)
    r.raise_for_status()
    return r.json()

def save_raw_csv(df: pd.DataFrame, kind: str, tag: str):
    path = RAW_DIR / f"{kind}_{tag}_{ts()}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path


In [27]:
def fetch_revenue_fmp(ticker: str, limit=40):
    """Return quarterly revenue for last 5y from FMP income-statement endpoint."""
    base = f"https://financialmodelingprep.com/api/v3//financial-growth/{ticker}"
    params = {"period": "quarter", "limit": limit, "apikey": FMP_API_KEY}
    js = http_get_json(base, params=params)
    df = pd.DataFrame(js)
    # Pick the first available from common revenue fields
    for col in ["revenue","totalRevenue","netRevenue"]:
        if col in df.columns:
            out = df[["date", col]].rename(columns={col: "revenue_total"}).copy()
            out["date"] = to_quarter_end(out["date"])
            out["revenue_total"] = pd.to_numeric(out["revenue_total"], errors="coerce")
            out = (out.dropna(subset=["revenue_total"])
                      .drop_duplicates(subset=["date"])
                      .sort_values("date"))
            return within_5y(out)
    raise ValueError(f"No revenue column found in FMP response for {ticker}")

# Pull all banks
frames = []
for bank, tkr in BANK_TICKERS.items():
    df = fetch_revenue_fmp(tkr, limit=40)
    df["bank"] = bank
    df["ticker"] = tkr
    frames.append(df[["bank","ticker","date","revenue_total"]])

rev_all = (pd.concat(frames, ignore_index=True)
             .drop_duplicates(subset=["bank","date"])
             .sort_values(["bank","date"]))
save_raw_csv(rev_all, "revenue_actuals_fmp", "banks5_all")
rev_all.tail()


HTTPError: 403 Client Error: Forbidden for url: https://financialmodelingprep.com/api/v3/financial-growth/JPM?period=quarter&limit=40&apikey=fOEWP2xR6ohtGmStvhnU1IPfj1pwGWk3

In [24]:
import yfinance as yf

def fetch_revenue_yf(ticker, limit=40):
    ticker_obj = yf.Ticker(ticker)
    df = ticker_obj.quarterly_financials.T
    df = df.reset_index().rename(columns={"index": "date"})
    df["ticker"] = ticker
    return df.head(limit)


In [28]:
# --- imports & setup ---
import os, time, datetime as dt, pandas as pd, requests
from pathlib import Path

try:
    import yfinance as yf
except ImportError:
    %pip install yfinance -q
    import yfinance as yf

FMP_API_KEY = os.getenv("FMP_API_KEY", "").strip()
DATA_DIR_RAW = Path("project/revenue_risk_model/data/raw")
DATA_DIR_RAW.mkdir(parents=True, exist_ok=True)

# --- tiny helper: robust GET that doesn't crash on 403/5xx ---
def http_get_json(url, params=None, timeout=30, retries=2, backoff=1.5):
    params = params or {}
    for attempt in range(retries + 1):
        r = requests.get(url, params=params, timeout=timeout)
        # hard fail on auth/forbidden but don't raise; let caller decide
        if r.status_code == 200:
            try:
                return r.json()
            except Exception:
                return None
        if r.status_code in (401, 403):
            # no permission / forbidden — don't retry endlessly
            return {"__error__": f"{r.status_code} {r.reason}", "__status__": r.status_code, "__url__": r.url}
        if r.status_code >= 500 and attempt < retries:
            time.sleep(backoff ** attempt)
            continue
        # other 4xx — give up
        return {"__error__": f"{r.status_code} {r.reason}", "__status__": r.status_code, "__url__": r.url}

# --- one place to parse 'revenue' from multiple possible keys ---
REV_KEYS = [
    "revenue", "totalRevenue", "total_revenue",
    "Revenue", "Total Revenue", "sales", "Sales", "totalSales"
]

def coerce_revenue_column(df):
    for k in REV_KEYS:
        if k in df.columns:
            return df.rename(columns={k: "revenue"})
    # fallback: try case-insensitive contains "revenue"
    for c in df.columns:
        if "revenue" in c.lower():
            return df.rename(columns={c: "revenue"})
    return df  # no rename

# --- main fetcher: FMP (two tries) -> yfinance fallback ---
def fetch_revenue_any(ticker: str, limit: int = 40) -> pd.DataFrame:
    rows = []
    source_used = None

    # 1) FMP modern endpoint
    if FMP_API_KEY:
        base = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
        params = {"period": "quarter", "limit": limit, "apikey": FMP_API_KEY}
        js = http_get_json(base, params=params)
        if isinstance(js, list) and js:
            df = pd.DataFrame(js)
            # Common FMP column is "date" (ISO yyyy-mm-dd); revenue may be "revenue"
            if "date" in df.columns:
                df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = coerce_revenue_column(df)
            if "revenue" in df.columns:
                out = df[["date", "revenue"]].copy()
                source_used = "fmp:v3"
                out["source"] = source_used
                return out.sort_values("date").tail(limit).reset_index(drop=True)
        # if error payload signals forbidden, try legacy next
        # (we won't abort here—just cascade)
    
        # 2) FMP legacy-ish variant (sometimes available on certain plans)
        base2 = f"https://financialmodelingprep.com/api/v3/income-statement-as-reported/{ticker}"
        params2 = {"period": "quarter", "limit": limit, "apikey": FMP_API_KEY}
        js2 = http_get_json(base2, params=params2)
        if isinstance(js2, list) and js2:
            df2 = pd.DataFrame(js2)
            # as-reported often has "fillingDate" or "date"
            date_col = "date" if "date" in df2.columns else ("fillingDate" if "fillingDate" in df2.columns else None)
            if date_col:
                df2["date"] = pd.to_datetime(df2[date_col], errors="coerce")
            df2 = coerce_revenue_column(df2)
            if "revenue" in df2.columns:
                out = df2[["date", "revenue"]].copy()
                source_used = "fmp:as-reported"
                out["source"] = source_used
                return out.sort_values("date").tail(limit).reset_index(drop=True)

    # 3) Yahoo Finance fallback (free, no key; quarterly statements)
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials.T  # index=quarters, columns=financial line items
    if qf is None or qf.empty:
        raise RuntimeError(f"No data from FMP or Yahoo for {ticker}.")
    qf = qf.reset_index().rename(columns={"index": "date"})
    # find a revenue-like column
    qf = coerce_revenue_column(qf)
    if "revenue" not in qf.columns:
        # Yahoo sometimes uses "Total Revenue"—coerce_revenue_column should’ve caught it.
        raise RuntimeError(f"Could not identify a revenue column for {ticker} from Yahoo.")
    qf["date"] = pd.to_datetime(qf["date"], errors="coerce")
    out = qf[["date", "revenue"]].sort_values("date").tail(limit).reset_index(drop=True)
    out["source"] = "yfinance"
    return out

# --- your loop (BANK_TICKERS example) ---
BANK_TICKERS = {
    "JPMorgan": "JPM",
    "Bank of America": "BAC",
    "Citigroup": "C",
    "Goldman Sachs": "GS",
    "Morgan Stanley": "MS",
    "Deutsche Bank": "DB",
}

frames = []
for bank, tkr in BANK_TICKERS.items():
    try:
        df = fetch_revenue_any(tkr, limit=40)
        df["bank"] = bank
        df["ticker"] = tkr
        frames.append(df)
        print(f"OK {tkr}: {df['source'].iat[-1]} -> {len(df)} rows")
    except Exception as e:
        print(f"FAIL {tkr}: {e}")

if frames:
    all_rev = pd.concat(frames, ignore_index=True)
    # basic validation
    assert all_rev["date"].notna().all(), "Some dates failed to parse."
    assert all_rev["revenue"].notna().any(), "Revenue entirely missing."
    # save raw
    stamp = dt.datetime.now().strftime("%Y%m%d-%H%M")
    out_path = DATA_DIR_RAW / f"api_revenue_banks_{stamp}.csv"
    all_rev.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")
else:
    raise RuntimeError("No bank produced data.")


OK JPM: yfinance -> 7 rows
OK BAC: yfinance -> 6 rows
OK C: yfinance -> 6 rows
OK GS: yfinance -> 7 rows
OK MS: yfinance -> 6 rows
OK DB: yfinance -> 6 rows
Saved -> project\revenue_risk_model\data\raw\api_revenue_banks_20250825-1441.csv


In [29]:
import pandas as pd, numpy as np, datetime as dt
from pathlib import Path

RAW_PATH = Path(r"project/revenue_risk_model/data/raw")  # adjust if needed
latest = sorted(RAW_PATH.glob("api_revenue_banks_*.csv"))[-1]
df = pd.read_csv(latest, parse_dates=["date"])

# basic tidy
df["bank"]   = df["bank"].astype("category")
df["ticker"] = df["ticker"].astype("category")
df["source"] = df["source"].astype("category")
df = df.sort_values(["ticker","date"]).reset_index(drop=True)

# validations
report = {}
report["rows"] = len(df)
report["cols"] = df.shape[1]
report["nulls_by_col"] = df.isna().sum().to_dict()
report["dupe_rows"] = int(df.duplicated(["ticker","date"]).sum())
report["date_monotonic_by_ticker"] = (
    df.groupby("ticker")["date"].apply(lambda s: s.is_monotonic_increasing).to_dict()
)
# quarterly gaps (should be near 90d deltas)
def _gap_okay(s):
    s = s.dropna().sort_values()
    if len(s) < 2: return True
    deltas = s.diff().dropna().dt.days.values
    # allow 60–120 days as “quarter-ish”
    return bool(np.all((deltas >= 60) & (deltas <= 120)))
report["quarterly_spacing_ok"] = (
    df.groupby("ticker")["date"].apply(_gap_okay).to_dict()
)

pd.DataFrame({
    "ticker": df["ticker"].cat.categories,
    "n_rows": [len(df[df["ticker"]==t]) for t in df["ticker"].cat.categories],
    "monotonic": [report["date_monotonic_by_ticker"][t] for t in df["ticker"].cat.categories],
    "q_spacing_ok": [report["quarterly_spacing_ok"][t] for t in df["ticker"].cat.categories],
})


,ticker,n_rows,monotonic,q_spacing_ok
0,BAC,6,True,True
1,C,6,True,True
2,DB,6,True,True
3,GS,7,True,True
4,JPM,7,True,True
5,MS,6,True,True


In [30]:
assert report["dupe_rows"] == 0, "Duplicate (ticker,date) pairs found."
assert df["date"].notna().all(), "Some date values failed to parse."
assert df["revenue"].notna().any(), "Revenue is entirely NA."
print("Validation ✅:", report)


Validation ✅: {'rows': 38, 'cols': 5, 'nulls_by_col': {'date': 0, 'revenue': 8, 'source': 0, 'bank': 0, 'ticker': 0}, 'dupe_rows': 0, 'date_monotonic_by_ticker': {'BAC': True, 'C': True, 'DB': True, 'GS': True, 'JPM': True, 'MS': True}, 'quarterly_spacing_ok': {'BAC': True, 'C': True, 'DB': True, 'GS': True, 'JPM': True, 'MS': True}}


In [36]:
# ensures parsers for pandas.read_html and BeautifulSoup
import importlib, sys, subprocess
def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ("lxml", "html5lib", "beautifulsoup4"):
    ensure(pkg)

print("Deps OK: lxml, html5lib, bs4")


Deps OK: lxml, html5lib, bs4


In [37]:
import os, re, datetime as dt
from pathlib import Path
import pandas as pd, requests

# target + output
URL = "https://en.wikipedia.org/wiki/List_of_largest_banks"
DATA_DIR_RAW = Path(os.getenv("DATA_DIR_RAW", "data/raw"))
DATA_DIR_RAW.mkdir(parents=True, exist_ok=True)
stamp = dt.datetime.now().strftime("%Y%m%d-%H%M")

# 1) fetch & parse with lxml (avoids html5lib dependency flakiness)
html = requests.get(URL, timeout=30).text
tables = pd.read_html(html, flavor="lxml")

# 2) pick a table that has 'Rank' or 'Assets' in the headers
pat = re.compile(r"(Rank|Assets|Total assets|Market)", re.I)
chosen = None
for t in tables:
    cols = [str(c) for c in t.columns]
    if any(pat.search(c) for c in cols):
        chosen = t
        break
assert chosen is not None, "Still couldn't find a suitable table on the page."

df = chosen.copy()

# 3) tidy headers and drop unnamed columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [" ".join([str(x) for x in tup if str(x) != "nan"]).strip() for tup in df.columns]
df.columns = (pd.Series(df.columns)
              .astype(str)
              .str.replace(r"\[\d+\]", "", regex=True)  # footnotes like [1]
              .str.replace("\xa0", " ")
              .str.strip())
df = df.loc[:, ~df.columns.str.contains("^Unnamed", case=False, regex=True)]
df = df.dropna(how="all")

# 4) light numeric coercion (no boolean indexers -> no alignment errors)
for c in df.columns:
    s = df[c].astype(str)
    if s.str.contains(r"\d", na=False).mean() > 0.5:      # looks numeric-ish
        s = (s.str.replace("\u2212", "-", regex=False)     # minus sign
               .str.replace(r"[^\d\.\-]", "", regex=True)) # strip units/commas
        # convert; if it's not really numeric it'll become NaN, which is fine
        df[c] = pd.to_numeric(s, errors="ignore")

# 5) quick validations (Stage-04 rubric style)
val = {
    "shape": df.shape,
    "nulls_top": df.isna().sum().sort_values(ascending=False).head(5).to_dict()
}
print("Validation ✅", val)

# 6) save raw
out_path = DATA_DIR_RAW / f"scrape_wiki_largest_banks_{stamp}.csv"
df.to_csv(out_path, index=False)
print(f"Scraped ✅ -> {out_path}")
df.head(3)


ValueError: No tables found matching regex '.+'

In [34]:
import re, os, datetime as dt
import pandas as pd, requests
from bs4 import BeautifulSoup
from pathlib import Path

# env-driven raw dir (works with Stage-05 later too)
DATA_DIR_RAW = Path(os.getenv("DATA_DIR_RAW", "data/raw"))
DATA_DIR_RAW.mkdir(parents=True, exist_ok=True)
stamp = dt.datetime.now().strftime("%Y%m%d-%H%M")

# 1) Candidate URLs (we’ll try each until something matches)
CANDIDATES = [
    # Wikipedia largest banks — structure changes often
    ("wiki_largest_banks", "https://en.wikipedia.org/wiki/List_of_largest_banks", r"Assets|Market|Rank"),
    # FDIC failed bank list — very stable government table
    ("fdic_failed_banks", "https://www.fdic.gov/resources/resolutions/bank-failures/failed-bank-list/", r"Bank|City|State|Closing|Acquiring"),
    # BIS list of jurisdictions (as a generic backstop example w/ simple table)
    ("wiki_bank_capital", "https://en.wikipedia.org/wiki/Capital_requirement", r"Country|Jurisdiction|Ratio|Requirement"),
]

def fetch_tables(url: str):
    """Return list of DataFrames using pandas read_html with a permissive parser."""
    # try direct read_html first (handles many cases)
    try:
        return pd.read_html(url, flavor=None)
    except Exception:
        # fall back: get HTML ourselves and read_html on the string
        html = requests.get(url, timeout=30).text
        return pd.read_html(html)

def first_matching_table(dfs, regex_pat: str):
    pat = re.compile(regex_pat, flags=re.I)
    for df in dfs:
        # normalize headers to strings
        cols = [str(c) for c in df.columns]
        if any(pat.search(c) for c in cols):
            return df
    return None

def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    # drop "Unnamed" columns, strip spaces/newlines, remove footnote markers like [1]
    df = df.loc[:, ~pd.Series(df.columns).astype(str).str.contains("^Unnamed", flags=re.I, regex=True)].copy()
    df.columns = (
        pd.Series(df.columns)
        .astype(str)
        .str.replace(r"\[\d+\]", "", regex=True)
        .str.replace(r"\xa0", " ", regex=False)
        .str.strip()
    )
    # trim strings in cells
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = (
                df[c].astype(str)
                .str.replace(r"\[\d+\]", "", regex=True)
                .str.replace("\xa0", " ")
                .str.strip()
                .replace({"": pd.NA})
            )
    return df

def try_scrape_candidates(candidates):
    errors = []
    for slug, url, match in candidates:
        try:
            tables = fetch_tables(url)
            df = first_matching_table(tables, match)
            if df is None:
                errors.append(f"{slug}: no matching columns for /{match}/")
                continue
            df = clean_table(df)
            # light numeric coercion for any column that looks number-ish
            for c in df.columns:
                if df[c].astype(str).str.contains(r"\d", na=False).mean() > 0.5:
                    df[c] = (
                        df[c].astype(str)
                        .str.replace(r"[^\d\.\-]", "", regex=True)
                        .replace({"": pd.NA})
                    )
                    # only convert if majority numeric
                    mask_num = df[c].astype(str).str.fullmatch(r"-?\d+(\.\d+)?", na=True)
                    if mask_num.mean() > 0.5:
                        df[c] = pd.to_numeric(df[c], errors="coerce")
            df["__source_url"] = url
            df["__source_slug"] = slug
            return df, slug
        except Exception as e:
            errors.append(f"{slug}: {e}")
            continue
    raise AssertionError("No suitable table found. Tried:\n- " + "\n- ".join(errors))

scrape_df, which = try_scrape_candidates(CANDIDATES)
out_path = DATA_DIR_RAW / f"scrape_{which}_{stamp}.csv"
scrape_df.to_csv(out_path, index=False)
print(f"Scraped ✅ {which} -> {out_path}  shape={scrape_df.shape}")
scrape_df.head(3)


AssertionError: No suitable table found. Tried:
- wiki_largest_banks: Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.
- fdic_failed_banks: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).
- wiki_bank_capital: Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.

In [33]:
import requests, pandas as pd
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_largest_banks"
html = requests.get(url, timeout=30).text
soup = BeautifulSoup(html, "lxml")

# pick the first table that mentions Assets or Market cap
tables = soup.select("table.wikitable")
df_tables = []
for tb in tables:
    df_try = pd.read_html(str(tb))[0]
    if any(c for c in df_try.columns if "Assets" in str(c) or "Market" in str(c)):
        df_tables.append(df_try)

assert df_tables, "No suitable table found."
scrape_df = df_tables[0].copy()

# light validation: drop empty cols, keep text cols, coerce any numeric-ish
scrape_df = scrape_df.loc[:, ~scrape_df.columns.astype(str).str.contains("^Unnamed")]
for c in scrape_df.columns:
    # try to coerce numeric columns (strip symbols)
    if scrape_df[c].astype(str).str.contains(r"\d").any():
        scrape_df[c] = (
            scrape_df[c].astype(str)
            .str.replace(r"[^\d\.\-]", "", regex=True)
            .replace({"": pd.NA})
        )
        # only convert if >50% look numeric
        mask_num = scrape_df[c].str.fullmatch(r"-?\d+(\.\d+)?", na=True)
        if mask_num.mean() > 0.5:
            scrape_df[c] = pd.to_numeric(scrape_df[c], errors="coerce")

# save raw scrape
scrape_out = DATA_DIR_RAW / f"scrape_wiki_largest_banks_{stamp}.csv"
scrape_df.to_csv(scrape_out, index=False)
print("Scraped ->", scrape_out, "shape:", scrape_df.shape)


AssertionError: No suitable table found.